# Score de propensão a apostar — notebook completo

Regressão logística sobre variáveis socioeconômicas, convertida em scorecard de pontos,
com análise de ponto de corte.

**Dados fictícios.** A base é gerada por simulação neste próprio notebook, calibrada a
prevalências publicadas pelo DataSenado (jun/2024), pelo Banco Central (EE119, ago/2024)
e pelo PoderData (set/2025).

Rode as células de cima para baixo. Semente fixa: os números saem idênticos aos do relatório.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

SEED = 20260831
rng = np.random.default_rng(SEED)
ALVO = "apostador"
print("pronto")

---
## 1. Gerar a base sintética

A população vem de marginais aproximadas do Brasil adulto. As dependências são construídas
com estrutura realista: renda log-normal deslocada por escolaridade, ocupação, idade e
região; benefício social com probabilidade caindo com a renda; ocupação condicionada à idade.

In [ ]:
SEXO = (["Masculino", "Feminino"], [0.48, 0.52])
FAIXA_IDADE = (["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
               [0.14, 0.22, 0.21, 0.17, 0.14, 0.12])
IDADE_LIM = {"18-24": (18, 24), "25-34": (25, 34), "35-44": (35, 44),
             "45-54": (45, 54), "55-64": (55, 64), "65+": (65, 82)}
ESCOLARIDADE = (["Fundamental incompleto", "Fundamental completo", "Medio completo",
                 "Superior incompleto", "Superior completo"], [0.28, 0.13, 0.34, 0.09, 0.16])
REGIAO = (["Sudeste", "Nordeste", "Sul", "Norte", "Centro-Oeste"],
          [0.42, 0.27, 0.14, 0.09, 0.08])
PORTE = (["Capital", "Regiao metropolitana", "Interior medio", "Interior pequeno"],
         [0.21, 0.19, 0.32, 0.28])
ESTADO_CIVIL = (["Solteiro", "Casado ou uniao estavel", "Divorciado", "Viuvo"],
                [0.40, 0.46, 0.09, 0.05])
OCUPACAO = (["CLT", "Autonomo ou informal", "Servidor publico", "Empresario",
             "Desempregado", "Aposentado"], [0.33, 0.30, 0.06, 0.05, 0.10, 0.16])

def sorteia(par, n, gen):
    return gen.choice(par[0], size=n, p=np.array(par[1]) / sum(par[1]))

In [ ]:
def monta_populacao(n, gen, drift=False):
    faixa_p = np.array(FAIXA_IDADE[1], dtype=float)
    ocup_p = np.array(OCUPACAO[1], dtype=float)
    if drift:                                   # a safra futura vem mais jovem e mais informal
        faixa_p = faixa_p * np.array([1.25, 1.15, 1.00, 0.92, 0.88, 0.85])
        ocup_p = ocup_p * np.array([0.92, 1.20, 1.00, 1.00, 1.10, 0.90])
    faixa_p /= faixa_p.sum(); ocup_p /= ocup_p.sum()

    faixa = gen.choice(FAIXA_IDADE[0], size=n, p=faixa_p)
    idade = np.array([gen.integers(*IDADE_LIM[f]) + 1 for f in faixa])
    df = pd.DataFrame({
        "id_pessoa": [f"P{i+1:06d}" for i in range(n)],
        "sexo": sorteia(SEXO, n, gen), "idade": idade, "faixa_idade": faixa,
        "escolaridade": sorteia(ESCOLARIDADE, n, gen),
        "regiao": sorteia(REGIAO, n, gen),
        "porte_municipio": sorteia(PORTE, n, gen),
        "estado_civil": sorteia(ESTADO_CIVIL, n, gen)})

    ocup = gen.choice(OCUPACAO[0], size=n, p=ocup_p)
    idoso = (df["idade"] >= 65).values
    ocup[idoso] = gen.choice(["Aposentado", "Autonomo ou informal", "CLT"],
                             size=int(idoso.sum()), p=[0.80, 0.14, 0.06])
    jovem = (df["idade"] < 25).values
    ocup[jovem] = gen.choice(["CLT", "Autonomo ou informal", "Desempregado", "Servidor publico"],
                             size=int(jovem.sum()), p=[0.36, 0.34, 0.26, 0.04])
    df["ocupacao"] = ocup

    base_esc = df["escolaridade"].map({"Fundamental incompleto": -0.42,
        "Fundamental completo": -0.22, "Medio completo": 0.0,
        "Superior incompleto": 0.20, "Superior completo": 0.70})
    base_ocup = df["ocupacao"].map({"CLT": 0.10, "Autonomo ou informal": -0.18,
        "Servidor publico": 0.45, "Empresario": 0.55, "Desempregado": -0.75,
        "Aposentado": -0.10})
    base_idade = np.clip((df["idade"] - 18) / 30, 0, 1.4) * 0.30
    base_reg = df["regiao"].map({"Sudeste": 0.10, "Sul": 0.08, "Centro-Oeste": 0.05,
                                 "Nordeste": -0.15, "Norte": -0.12})
    log_sm = np.log(2.05) + base_esc + base_ocup + base_idade + base_reg + gen.normal(0, 0.70, n)
    renda = np.clip(np.exp(log_sm), 0.0, 40).round(2)
    df["renda_sm"] = renda
    df["faixa_renda"] = pd.cut(renda, [-0.01, 1, 2, 3, 5, 100],
        labels=["Ate 1 SM", "1 a 2 SM", "2 a 3 SM", "3 a 5 SM", "Acima de 5 SM"]).astype(str)

    p_ben = 1 / (1 + np.exp(2.2 * (renda - 1.3)))
    df["recebe_beneficio_social"] = (gen.random(n) < p_ben * 0.85).astype(int)
    df["qtd_dependentes"] = np.minimum(
        gen.poisson(np.where(df["estado_civil"] == "Casado ou uniao estavel", 1.5, 0.6)), 6)
    return df

### O logit verdadeiro

Os coeficientes abaixo foram escolhidos para reproduzir as prevalências publicadas. Como a
base é sintética, existe uma **verdade de terreno** — dá para perguntar depois se o modelo
recuperou o que estava lá.

In [ ]:
COEF = {
    "sexo": {"Masculino": 0.28, "Feminino": -0.26},
    "faixa_idade": {"18-24": 0.38, "25-34": 0.22, "35-44": 0.02,
                    "45-54": -0.15, "55-64": -0.34, "65+": -0.62},
    "escolaridade": {"Fundamental incompleto": 0.16, "Fundamental completo": 0.12,
                     "Medio completo": 0.00, "Superior incompleto": -0.08,
                     "Superior completo": -0.28},
    "faixa_renda": {"Ate 1 SM": 0.18, "1 a 2 SM": 0.10, "2 a 3 SM": -0.02,
                    "3 a 5 SM": -0.14, "Acima de 5 SM": -0.32},
    "ocupacao": {"CLT": 0.00, "Autonomo ou informal": 0.16, "Servidor publico": -0.14,
                 "Empresario": -0.04, "Desempregado": 0.18, "Aposentado": -0.16},
    "regiao": {"Norte": 0.16, "Nordeste": 0.10, "Centro-Oeste": 0.00,
               "Sudeste": -0.05, "Sul": -0.10},
    "porte_municipio": {"Capital": 0.08, "Regiao metropolitana": 0.04,
                        "Interior medio": -0.02, "Interior pequeno": -0.08},
    "estado_civil": {"Solteiro": 0.05, "Casado ou uniao estavel": -0.02,
                     "Divorciado": 0.02, "Viuvo": -0.04},   # fraca de proposito
}

def logito(df):
    z = np.zeros(len(df))
    for col, mapa in COEF.items():
        z += df[col].map(mapa).astype(float).values
    return z + 0.24 * df["recebe_beneficio_social"].values + 0.010 * df["qtd_dependentes"].values

def calibra_intercepto(z, prev_alvo):
    """Bissecao ate a prevalencia media bater o alvo."""
    lo, hi = -12.0, 12.0
    for _ in range(200):
        meio = (lo + hi) / 2
        if (1 / (1 + np.exp(-(meio + z)))).mean() > prev_alvo: hi = meio
        else: lo = meio
    return (lo + hi) / 2

def gera(n, prev, gen, drift=False):
    df = monta_populacao(n, gen, drift=drift)
    z = logito(df)
    p = 1 / (1 + np.exp(-(calibra_intercepto(z, prev) + z)))
    df[ALVO] = (gen.random(n) < p).astype(int)
    return df

dev = gera(60_000, 0.13, rng)
oot = gera(20_000, 0.16, rng, drift=True)
print(f"desenvolvimento {len(dev):,} | prevalencia {dev[ALVO].mean():.4f}")
print(f"safra futura    {len(oot):,} | prevalencia {oot[ALVO].mean():.4f}")

### A conferência que valida a geração

Gerar não basta: é preciso mostrar que a base saiu onde deveria.

In [ ]:
ap = dev[dev[ALVO] == 1]
print("PREVALENCIA GERAL      %.1f%%   (fonte: 13%%)" % (dev[ALVO].mean() * 100))
print("%% de homens entre apostadores  %.1f%%   (fonte: 62%%)"
      % ((ap["sexo"] == "Masculino").mean() * 100))
print("%% com menos de 40 anos         %.1f%%   (fonte: 56%%)" % ((ap["idade"] < 40).mean() * 100))
print("%% ate 2 SM                     %.1f%%   (fonte: 52%%)"
      % (ap["faixa_renda"].isin(["Ate 1 SM", "1 a 2 SM"]).mean() * 100))
print()
print("prevalencia entre beneficiarios %.1f%%  (fonte BCB: 17%%)"
      % (dev.loc[dev.recebe_beneficio_social == 1, ALVO].mean() * 100))
dev.groupby("faixa_idade")[ALVO].agg(n="size", taxa="mean").assign(
    taxa=lambda d: (d.taxa * 100).round(1))

---
## 2. WOE e Information Value

**Convenção adotada** (declarada de propósito, porque varia por autor):

`WOE = ln( P(faixa | apostador) / P(faixa | não apostador) )`

WOE positivo = faixa com mais apostadores que a média. Coeficiente positivo = aumenta a
propensão. O score cresce com a propensão.

In [ ]:
for d in (dev, oot):
    d["faixa_dependentes"] = pd.cut(d["qtd_dependentes"], [-0.1, 0, 1, 2, 10],
                                    labels=["0", "1", "2", "3 ou mais"]).astype(str)
    d["flag_beneficio"] = d["recebe_beneficio_social"].map({0: "Nao recebe", 1: "Recebe"})

VARIAVEIS = ["sexo", "faixa_idade", "escolaridade", "faixa_renda", "ocupacao", "regiao",
             "porte_municipio", "estado_civil", "faixa_dependentes", "flag_beneficio"]

treino, teste = train_test_split(dev, test_size=0.30, random_state=SEED, stratify=dev[ALVO])
print(f"treino {len(treino):,} | teste {len(teste):,}")

In [ ]:
def tabela_woe(df, var, alvo=ALVO):
    g = df.groupby(var)[alvo].agg(n="size", eventos="sum").reset_index()
    g["nao_eventos"] = g["n"] - g["eventos"]
    tot_e, tot_ne = g["eventos"].sum(), g["nao_eventos"].sum()
    g["taxa"] = g["eventos"] / g["n"]
    # a correcao de 0,5 evita log de zero em faixa sem evento
    g["dist_evento"] = (g["eventos"] + 0.5) / (tot_e + 0.5 * len(g))
    g["dist_nao_evento"] = (g["nao_eventos"] + 0.5) / (tot_ne + 0.5 * len(g))
    g["woe"] = np.log(g["dist_evento"] / g["dist_nao_evento"])
    g["iv_parcial"] = (g["dist_evento"] - g["dist_nao_evento"]) * g["woe"]
    g["variavel"] = var
    return g.rename(columns={var: "faixa"})

tabs = {v: tabela_woe(treino, v) for v in VARIAVEIS}
tabs["faixa_idade"][["faixa", "n", "eventos", "taxa", "woe", "iv_parcial"]].round(4)

In [ ]:
def forca(iv):
    return ("Sem poder" if iv < 0.02 else "Fraco" if iv < 0.10 else
            "Medio" if iv < 0.30 else "Forte" if iv < 0.50 else "Suspeito")

iv = pd.DataFrame({"variavel": VARIAVEIS,
                   "iv": [tabs[v]["iv_parcial"].sum() for v in VARIAVEIS]}
                  ).sort_values("iv", ascending=False).reset_index(drop=True)
iv["forca"] = iv["iv"].apply(forca)
iv.round(4)

In [ ]:
CORTE_IV = 0.02
selecionadas = iv.loc[iv["iv"] >= CORTE_IV, "variavel"].tolist()
print("selecionadas:", selecionadas)
print("descartadas :", iv.loc[iv["iv"] < CORTE_IV, "variavel"].tolist())

def aplica_woe(df, variaveis):
    s = pd.DataFrame(index=df.index)
    for v in variaveis:
        s[f"woe_{v}"] = df[v].map(dict(zip(tabs[v]["faixa"], tabs[v]["woe"]))).fillna(0.0)
    return s

Xtr, Xte, Xoot = (aplica_woe(d, selecionadas) for d in (treino, teste, oot))
ytr, yte, yoot = treino[ALVO].values, teste[ALVO].values, oot[ALVO].values

---
## 3. Redundância: correlação e VIF

O IV avalia cada variável isolada. Duas podem ter IV alto e dizer a mesma coisa.

In [ ]:
def vif(X):
    linhas = []
    for c in X.columns:
        outras = [o for o in X.columns if o != c]
        r2 = LinearRegression().fit(X[outras], X[c]).score(X[outras], X[c])
        linhas.append({"variavel": c, "vif": 1 / max(1e-9, 1 - r2)})
    return pd.DataFrame(linhas).sort_values("vif", ascending=False)

cm = Xtr.corr().where(~np.eye(len(Xtr.columns), dtype=bool)).abs().stack()
print("maior correlacao entre WOE:")
print(cm.sort_values(ascending=False).head(3).round(3))
print()
vif(Xtr).round(2)

---
## 4. A regressão logística

Erros-padrão pela matriz de informação de Fisher — não é preciso `statsmodels`.

In [ ]:
mod = LogisticRegression(C=np.inf, max_iter=2000).fit(Xtr, ytr)

Xd = np.column_stack([np.ones(len(Xtr)), Xtr.values])
p_tr = mod.predict_proba(Xtr)[:, 1]
cov = np.linalg.inv(Xd.T @ (Xd * (p_tr * (1 - p_tr))[:, None]))
se = np.sqrt(np.diag(cov))
coefs = np.concatenate([mod.intercept_, mod.coef_[0]])

pd.DataFrame({"termo": ["intercepto"] + list(Xtr.columns), "coeficiente": coefs,
              "erro_padrao": se, "z": coefs / se,
              "p_valor": 2 * (1 - norm.cdf(np.abs(coefs / se)))}).round(4)

---
## 5. Do coeficiente ao ponto

**PDO = 50**: a cada 50 pontos, a chance de ser apostador dobra.
**Score base = 500**, ancorado na prevalência da população.

Atenção: aqui **score maior = propensão maior** — o inverso da convenção de crédito.

In [ ]:
PDO, SCORE_BASE = 50, 500
odds_base = treino[ALVO].mean() / (1 - treino[ALVO].mean())
fator = PDO / np.log(2)
offset = SCORE_BASE - fator * np.log(odds_base)
alpha, beta = mod.intercept_[0], mod.coef_[0]
n_var = len(selecionadas)
print(f"fator {fator:.4f} | offset {offset:.4f}")

scorecard = []
for k, v in enumerate(selecionadas):
    t = tabs[v].copy()
    t["coeficiente"] = beta[k]
    t["pontos"] = ((offset + fator * alpha) / n_var + fator * beta[k] * t["woe"]).round(1)
    scorecard.append(t)
scorecard = pd.concat(scorecard, ignore_index=True)

def score_de(df):
    s = np.zeros(len(df))
    for v in selecionadas:
        sub = scorecard[scorecard["variavel"] == v]
        s += df[v].map(dict(zip(sub["faixa"], sub["pontos"]))).fillna(0).values
    return s

for d, X in ((treino, Xtr), (teste, Xte), (oot, Xoot)):
    d["score"] = score_de(d)
    d["prob"] = mod.predict_proba(X)[:, 1]

# o score em pontos tem que ser equivalente a probabilidade do modelo
rec = 1 / (1 + np.exp(-((teste["score"] - offset) / fator)))
print(f"erro maximo score -> probabilidade: {np.abs(rec - teste['prob']).max():.2e}")
scorecard[["variavel", "faixa", "n", "taxa", "woe", "pontos"]].head(12).round(4)

### Um caso concreto

Homem, 25-34 anos, 1 a 2 SM, autônomo, ensino médio completo, recebe benefício social.

In [ ]:
pessoa = {"sexo": "Masculino", "faixa_idade": "25-34", "faixa_renda": "1 a 2 SM",
          "ocupacao": "Autonomo ou informal", "escolaridade": "Medio completo",
          "flag_beneficio": "Recebe"}
total = 0
for v, f in pessoa.items():
    p = float(scorecard[(scorecard.variavel == v) & (scorecard.faixa == f)]["pontos"].iloc[0])
    print(f"  {v:<18} {f:<24} {p:>7.1f}")
    total += p
prob = 1 / (1 + np.exp(-((total - offset) / fator)))
print(f"\n  SCORE {total:.1f}  ->  probabilidade {prob:.2%}  (media da base: {dev[ALVO].mean():.2%})")

---
## 6. Desempenho

In [ ]:
def ks(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))

def metricas(y, p):
    auc = roc_auc_score(y, p)
    return {"n": len(y), "prevalencia": np.mean(y), "auc": auc, "gini": 2 * auc - 1,
            "ks": ks(y, p), "brier": brier_score_loss(y, p)}

pd.DataFrame({n: metricas(y, d["prob"].values) for n, y, d in
              (("treino", ytr, treino), ("teste", yte, teste), ("oot", yoot, oot))}).T.round(4)

In [ ]:
def tabela_decis(d, y, k=10):
    q = pd.qcut(d["score"], k, labels=False, duplicates="drop")
    t = (pd.DataFrame({"faixa": q, "y": y, "score": d["score"]})
         .groupby("faixa").agg(n=("y", "size"), eventos=("y", "sum"),
                               score_min=("score", "min"), score_max=("score", "max"))
         .sort_index(ascending=False).reset_index(drop=True))
    t.index = range(1, len(t) + 1)
    t["taxa"] = t["eventos"] / t["n"]
    t["lift"] = t["taxa"] / y.mean()
    t["ac_eventos"] = t["eventos"].cumsum() / t["eventos"].sum()
    t["ac_nao_eventos"] = (t["n"] - t["eventos"]).cumsum() / (t["n"] - t["eventos"]).sum()
    t["ks"] = (t["ac_eventos"] - t["ac_nao_eventos"]).abs()
    return t

decis = tabela_decis(teste, yte)
decis.round(4)

---
## 7. Calibração — e o achado da safra futura

Ordenar bem não é acertar o nível. Aqui está o ponto mais interessante do estudo.

In [ ]:
def curva_calibracao(p, y, k=10):
    return (pd.DataFrame({"p": p, "y": y})
            .assign(faixa=lambda d: pd.qcut(d["p"], k, labels=False, duplicates="drop"))
            .groupby("faixa").agg(prevista=("p", "mean"), observada=("y", "mean"),
                                  n=("y", "size")).reset_index())

print("TESTE");        print(curva_calibracao(teste["prob"], yte).round(4).to_string(index=False))
print("\nSAFRA FUTURA"); print(curva_calibracao(oot["prob"], yoot).round(4).to_string(index=False))
print(f"\nmedia prevista {oot['prob'].mean():.4f} vs observada {yoot.mean():.4f}"
      f"  ->  desvio {oot['prob'].mean() - yoot.mean():+.4f}")

In [ ]:
def psi(base, comp, bins=10):
    cortes = np.percentile(base, np.linspace(0, 100, bins + 1))
    cortes[0], cortes[-1] = -np.inf, np.inf
    b = np.clip(np.histogram(base, cortes)[0] / len(base), 1e-6, None)
    c = np.clip(np.histogram(comp, cortes)[0] / len(comp), 1e-6, None)
    return float(np.sum((c - b) * np.log(c / b)))

print(f"PSI treino x teste        : {psi(treino['score'], teste['score']):.4f}")
print(f"PSI treino x safra futura : {psi(treino['score'], oot['score']):.4f}")
print(f"AUC teste {roc_auc_score(yte, teste['prob']):.4f} | "
      f"AUC safra {roc_auc_score(yoot, oot['prob']):.4f}")

**O diagnóstico:** discriminação preservada (AUC igual), distribuição estável (PSI baixo),
nível deslocado. Isso não é modelo quebrado — é modelo descalibrado. A correção é deslocar
o intercepto, não remodelar.

In [ ]:
logito_oot = np.log(oot["prob"] / (1 - oot["prob"]))
delta = brentq(lambda d: (1 / (1 + np.exp(-(logito_oot + d)))).mean() - yoot.mean(), -3, 3)
prob_recal = 1 / (1 + np.exp(-(logito_oot + delta)))

print(f"deslocamento de intercepto : {delta:+.4f}")
print(f"em pontos de score         : {delta * fator:+.1f}")
print(f"Brier  antes {brier_score_loss(yoot, oot['prob']):.4f} -> "
      f"depois {brier_score_loss(yoot, prob_recal):.4f}")
print(f"AUC    antes {roc_auc_score(yoot, oot['prob']):.4f} -> "
      f"depois {roc_auc_score(yoot, prob_recal):.4f}   (inalterado, como tem que ser)")

---
## 8. O ponto de corte

O corte não é escolha estatística: é escolha de negócio. A estatística mostra o que se
ganha e o que se perde.

In [ ]:
def tabela_cortes(d, y, cortes):
    linhas = []
    for c in cortes:
        sel = d["score"].values >= c
        tp = int((sel & (y == 1)).sum()); fp = int((sel & (y == 0)).sum())
        fn = int((~sel & (y == 1)).sum()); tn = int((~sel & (y == 0)).sum())
        prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn); esp = tn / max(1, tn + fp)
        linhas.append({"corte": c, "pct_base": sel.mean(), "precisao": prec, "recall": rec,
                       "especificidade": esp, "lift": prec / y.mean(),
                       "f1": 2 * prec * rec / max(1e-9, prec + rec),
                       "youden": rec + esp - 1})
    return pd.DataFrame(linhas)

cortes = tabela_cortes(teste, yte, np.arange(420, 601, 10))
cortes.round(4)

In [ ]:
melhor = cortes.loc[cortes["youden"].idxmax()]
print(f"corte estatistico (Youden = ponto do KS): {melhor['corte']:.0f}")
print()
for c, nome in ((480, "cobertura"), (510, "KS"), (550, "foco")):
    r = cortes[cortes["corte"] == c].iloc[0]
    print(f"{nome:>10} (corte {c}): aborda {r['pct_base']:5.1%} da base | "
          f"precisao {r['precisao']:5.1%} | captura {r['recall']:5.1%} | lift {r['lift']:.2f}")

**A ressalva que precisa acompanhar o número:** o corte foi calculado na amostra de teste,
com prevalência de 13,1%. Na safra futura, com 15,9%, o corte equivalente sobe cerca de
13 pontos — os mesmos da recalibração. **Cutoff não é constante:** ele acompanha a
prevalência, e revisá-lo faz parte do monitoramento.

---
## 9. Os gráficos

In [ ]:
PAL = {"azul": "#2a78d6", "laranja": "#eb6834", "verde": "#1baf7a",
       "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
       "grid": "#e1e0d9", "surface": "#fcfcfb"}
plt.rcParams.update({
    "figure.facecolor": PAL["surface"], "axes.facecolor": PAL["surface"],
    "text.color": PAL["ink"], "axes.labelcolor": PAL["ink2"],
    "xtick.color": PAL["muted"], "ytick.color": PAL["muted"],
    "axes.edgecolor": "#c3c2b7", "grid.color": PAL["grid"],
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.4))
for nome, d, y, cor in (("Treino", treino, ytr, PAL["azul"]),
                        ("Teste", teste, yte, PAL["laranja"]),
                        ("Safra futura", oot, yoot, PAL["verde"])):
    fpr, tpr, _ = roc_curve(y, d["prob"])
    a.plot(fpr, tpr, color=cor, lw=2, label=f"{nome} — AUC {roc_auc_score(y, d['prob']):.3f}")
a.plot([0, 1], [0, 1], color=PAL["muted"], lw=1, ls="--")
a.set_xlabel("1 - especificidade"); a.set_ylabel("Sensibilidade")
a.legend(loc="lower right"); a.grid(True, color=PAL["grid"]); a.set_axisbelow(True)
a.set_title("Curva ROC", loc="left", fontweight="bold")

bins = np.linspace(teste["score"].min(), teste["score"].max(), 44)
for classe, cor, rot in ((0, PAL["azul"], "Nao apostador"), (1, PAL["laranja"], "Apostador")):
    b.hist(teste.loc[teste[ALVO] == classe, "score"], bins=bins, density=True,
           color=cor, alpha=.55, label=rot)
b.axvline(510, color=PAL["ink2"], lw=1.6, ls="--")
b.set_xlabel("Score"); b.set_ylabel("Densidade"); b.legend(loc="upper left")
b.grid(True, color=PAL["grid"]); b.set_axisbelow(True)
b.set_title("Distribuicao do score por classe", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.4))
x = np.arange(1, len(decis) + 1)
a.bar(x, decis["taxa"], color=PAL["azul"], width=.72)
a.axhline(yte.mean(), color=PAL["laranja"], lw=1.6, ls="--")
a.set_xticks(x); a.set_xlabel("Decil (1 = maior propensao)"); a.set_ylabel("Taxa")
a.grid(True, axis="y", color=PAL["grid"]); a.set_axisbelow(True)
a.set_title("Taxa por decil", loc="left", fontweight="bold")

c = cortes
b.plot(c["corte"], c["recall"], color=PAL["azul"], lw=2, label="Recall")
b.plot(c["corte"], c["precisao"], color=PAL["laranja"], lw=2, label="Precisao")
b.plot(c["corte"], c["pct_base"], color=PAL["verde"], lw=2, label="% da base")
for corte in (480, 510, 550):
    b.axvline(corte, color=PAL["muted"], lw=1, ls=":")
b.set_xlabel("Ponto de corte"); b.legend(loc="center right")
b.grid(True, axis="y", color=PAL["grid"]); b.set_axisbelow(True)
b.set_title("O que se ganha e o que se perde", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 10. Exportar

O scorecard e as bases saem prontos para a planilha e para a próxima etapa.

In [ ]:
COLS = ["id_pessoa", "sexo", "idade", "faixa_idade", "escolaridade", "renda_sm",
        "faixa_renda", "ocupacao", "regiao", "porte_municipio", "estado_civil",
        "qtd_dependentes", "recebe_beneficio_social", ALVO]

scorecard.to_csv("scorecard_apostador.csv", index=False)
dev[COLS].to_csv("base_apostadores.csv", index=False)
oot[COLS].to_csv("safra_futura.csv", index=False)
teste[["id_pessoa", "score", "prob", ALVO]].to_csv("scores_teste.csv", index=False)
print("arquivos gravados")

---
## Limitações

1. **Dados fictícios.** Calibrados a prevalências reais, mas as correlações são as que impus.
2. **Poder preditivo baixo, e é do problema.** KS de 22 é o teto de um score socioeconômico.
   Trocar por gradient boosting renderia 1 ou 2 pontos ao custo da interpretabilidade.
3. **O sinal de verdade está no transacional** — Pix recorrente, horário, saldo caindo no fim
   do mês. Foi assim que o Banco Central enxergou o fenômeno.
4. **Risco de discriminação indireta.** Renda, escolaridade e benefício social são proxies de
   raça e classe no Brasil. Uso defensável: priorizar proteção e educação financeira.
   Uso indefensável: negar crédito, precificar ou limitar produto.
5. **Causalidade não foi tratada.** Deixei restritivo e endividamento fora de propósito:
   são provavelmente consequência de apostar, não causa.